In [ ]:
from pathlib import Path
import hashlib
import inspect
import json

from seisnetinsight.config import GridParameters
from seisnetinsight.data import load_events, load_stations, load_swd_wells
from seisnetinsight.grids import (
    compute_composite_index,
    compute_gap_grid,
    compute_subject_grids,
    compute_swd_grid,
    generate_grid,
    merge_grids,
)
import pandas as pd

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

def _cache_path(name: str) -> Path:
    fingerprints = {str(path): hashlib.sha256(Path(path).read_bytes()).hexdigest()
                    for path in (EVENTS_CSV, STATIONS_CSV, SWD_CSV)}
    implementation = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                      for path in Path(inspect.getfile(compute_composite_index)).parent.glob("*.py")}
    key = hashlib.sha256(json.dumps({"parameters": vars(PARAMS), "inputs": fingerprints, "implementation": implementation}, sort_keys=True).encode()).hexdigest()[:16]
    return CACHE_DIR / f"{name}_{key}.parquet"

def load_or_compute(name: str, compute_fn, *, overwrite=False):
    path = _cache_path(name)
    if path.exists() and not overwrite:
        return pd.read_parquet(path)
    df = compute_fn()
    df.to_parquet(path, index=False)
    return df

In [ ]:
bna_path = Path("../sample_files/midland.bna")
bna_bytes = bna_path.read_bytes()

EVENTS_CSV = '../sample_files/events_sample.csv'
STATIONS_CSV = '../sample_files/texnet_stations_2025.csv'
SWD_CSV = '../sample_files/swd_wells_sample.csv'

SWD_COLUMN_MAP = {
    "latitude": "latitude_wgs84",
    "longitude": "longitude_wgs84",
    "volume": "SUM_injected_liquid_BBL",
}

PARAMS = GridParameters(
    lons=(-102.9, -100.4),
    lats=(30.5, 34.0),
    grid_step=0.01,
    subject_primary_radius_km=4.0,
    subject_primary_min_stations=1,
    subject_primary_weight=0.3,
    subject_secondary_radius_km=10.0,
    subject_secondary_min_stations=1,
    subject_secondary_weight=0.4,
    gap_search_km=40.0,
    gap_target_angle_deg=90.0,
    weight_gap=0.1,
    swd_radius_km=25.0,
    weight_swd=0.2,
    half_time_years=5.0,
)

# Grids

In [ ]:
params = PARAMS
events_df, missing_events = load_events(EVENTS_CSV, warn=False)
stations_df, missing_stations = load_stations(STATIONS_CSV, warn=False)
swd_df, missing_swd = load_swd_wells(SWD_CSV, column_map=SWD_COLUMN_MAP, warn=False)

grid = generate_grid(params)

In [ ]:
subject_df = load_or_compute(
    "subject",
    lambda: compute_subject_grids(events_df, stations_df, grid, params),
    overwrite=False,
)

In [ ]:
gap_df = load_or_compute(
    "gap",
    lambda: compute_gap_grid(events_df, stations_df, grid, params),
    overwrite=False,
)

In [ ]:
swd_grid = load_or_compute(
    "swd",
    lambda: compute_swd_grid(swd_df, grid, params),
    overwrite=False,
)

In [ ]:
merged = merge_grids(subject_df, gap_df, swd_grid)
composite_grid_df = load_or_compute(
    "composite",
    lambda: compute_composite_index(
        merged,
        params,
    ),
    overwrite=False,
)

# Maps

In [ ]:
from seisnetinsight.legacy_maps import (
    render_legacy_contour,
    render_priority_clusters,
    LegacyMapConfig,
)
import matplotlib.pyplot as plt
%matplotlib inline
def show(fig):
    display(fig)
    plt.close(fig)

config = LegacyMapConfig(show_roads=True,
                         show_states=True,
                         show_cities=True, bna_label="Midland")
# all the figures can also use the composite_grid_df
# fig = render_legacy_contour(composite_grid_df, "subject_primary_weighted", params, config=config, bna_bytes=bna_bytes)
fig = render_legacy_contour(subject_df, "subject_primary_weighted", params, config=config, bna_bytes=bna_bytes)

In [ ]:
# all the figures can also use the composite_grid_df
# fig = render_legacy_contour(composite_grid_df, "subject_secondary_weighted", params, config=config, bna_bytes=bna_bytes)
fig = render_legacy_contour(subject_df, "subject_secondary_weighted", params, config=config, bna_bytes=bna_bytes)

In [ ]:
fig = render_legacy_contour(gap_df, "delta_gap90_weighted", params, config=config, bna_bytes=bna_bytes)

In [ ]:
fig = render_legacy_contour(composite_grid_df, "swd_volume_25km_bbl", params, config=config, bna_bytes=bna_bytes)

In [ ]:
fig = render_legacy_contour(composite_grid_df, "composite_index", params, config=config, bna_bytes=bna_bytes)

In [ ]:
from seisnetinsight.maps import classify_priority_clusters

priority_df = classify_priority_clusters(
    composite_grid_df,
    features=None,          # defaults to SUBJECT_PRIMARY, SUBJECT_SECONDARY, GAP, SWD
    n_clusters=4,
    n_init=50,
    apply_feature_weights=True,
    feature_weights={
        "subject_primary_weighted": params.subject_primary_weight,
        "subject_secondary_weighted": params.subject_secondary_weight,
        "delta_gap90_weighted": params.weight_gap,
        "context_value": params.weight_swd,
    },
)

fig = render_priority_clusters(
    priority_df,
    params,
    stations=stations_df,
    config=config,
    bna_bytes=bna_bytes,
)

In [ ]:
feature_weights={
        "subject_primary_weighted": params.subject_primary_weight,
        "subject_secondary_weighted": params.subject_secondary_weight,
        "delta_gap90_weighted": params.weight_gap,
        "context_value": params.weight_swd,
    }
feature_weights

In [ ]:
priority_df.head()

In [ ]:
priority_df[['delta_gap90_weighted', 'composite_index']].describe()

In [ ]:
count = (priority_df['delta_gap90_weighted'] > 208).sum()
print(f"Number of points with delta_gap90_weighted > 208: {count}")

In [ ]:
import seisnetinsight; print(seisnetinsight.__file__)

In [ ]:
# sklearn version
import sklearn
print(f'sklearn version: {sklearn.__version__}')

In [ ]:
# python version
import sys
print(f'python version: {sys.version}')